# Presidio dans RAGFlow — Démonstrateur technique

Ce notebook illustre les principes fondamentaux du masquage PII et de la réhydratation tels qu'implémentés dans le fork Eurelis de RAGFlow (`rag/llm/pii_masking.py`).  
Il est **décorrélé de RAGFlow** : seules les bibliothèques Presidio et spaCy sont nécessaires.

---

## Sommaire

1. [Installation & imports](#1-installation)
2. [Détection PII — AnalyzerEngine](#2-detection)
3. [Anonymisation — AnonymizerEngine](#3-anonymisation)
4. [Placeholders numérotés et cohérents](#4-placeholders)
5. [Masquage multi-messages avec état partagé](#5-multi-messages)
6. [Réhydratation des réponses](#6-rehydratation)
7. [Réhydratation en streaming (sliding-window buffer)](#7-streaming)
8. [Configuration : seuils, entités, actions MASK / BLOCK](#8-configuration)
9. [Recognizers personnalisés](#9-custom-recognizers)
10. [NER spaCy — détection contextuelle](#10-ner)
11. [Scénario complet bout-en-bout](#11-scenario)

---
## 1. Installation & imports <a id="1-installation"></a>

In [18]:
# Installation (à exécuter une seule fois)
%pip install presidio-analyzer presidio-anonymizer
%python -m spacy download en_core_web_sm
%python -m spacy download fr_core_news_sm

Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [2]:
import re
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

from presidio_analyzer import AnalyzerEngine, RecognizerResult
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

print("✅ Presidio importé avec succès")

✅ Presidio importé avec succès


---
## 2. Détection PII — AnalyzerEngine <a id="2-detection"></a>

`AnalyzerEngine` est le composant Presidio qui **détecte** les entités PII dans un texte.  
Il retourne une liste de `RecognizerResult` : type d'entité, position, score de confiance.

In [17]:
analyzer = AnalyzerEngine()

texte = "Bonjour, je suis Alice Martin, mon email est alice@example.com et mon téléphone le +33 6 12 34 56 78."

resultats = analyzer.analyze(
    text=texte,
    language="en",   # Presidio utilise 'en' même pour du texte partiellement FR sans modèle FR
    entities=["EMAIL_ADDRESS", "PHONE_NUMBER", "PERSON"],
)

print(f"Texte analysé : {texte!r}\n")
for r in sorted(resultats, key=lambda x: x.start):
    extrait = texte[r.start:r.end]
    print(f"  [{r.entity_type:20s}] pos={r.start:3d}-{r.end:3d}  score={r.score:.2f}  valeur={extrait!r}")

Texte analysé : 'Bonjour, je suis Alice Martin, mon email est alice@example.com et mon téléphone le +33 6 12 34 56 78.'

  [PERSON              ] pos=  0-  7  score=0.85  valeur='Bonjour'
  [PERSON              ] pos=  9- 29  score=0.85  valeur='je suis Alice Martin'
  [EMAIL_ADDRESS       ] pos= 45- 62  score=1.00  valeur='alice@example.com'
  [PHONE_NUMBER        ] pos= 83-100  score=0.75  valeur='+33 6 12 34 56 78'


### Entités disponibles par défaut

Presidio fournit des recognizers pour une quinzaine d'entités sans modèle NER :

In [5]:
entites_regex = [
    "EMAIL_ADDRESS", "PHONE_NUMBER", "IP_ADDRESS",
    "CREDIT_CARD", "IBAN_CODE", "US_SSN", "US_PASSPORT",
    "URL", "NRP", "DATE_TIME", "LOCATION", "PERSON",
]

textes_demo = {
    "EMAIL_ADDRESS": "Contactez support@eurelis.com pour toute question.",
    "PHONE_NUMBER":  "Appelez le +33 1 23 45 67 89 ou le 06.12.34.56.78.",
    "IP_ADDRESS":    "La requête provient de 192.168.1.42.",
    "CREDIT_CARD":   "Paiement par carte 4111 1111 1111 1111.",
    "IBAN_CODE":     "Virement vers FR76 3000 6000 0112 3456 7890 189.",
}

for entite, t in textes_demo.items():
    res = analyzer.analyze(text=t, language="en", entities=[entite])
    trouvees = [t[r.start:r.end] for r in res]
    print(f"  {entite:20s} → {trouvees}")

  EMAIL_ADDRESS        → ['support@eurelis.com']
  PHONE_NUMBER         → ['+33 1 23 45 67 89', '06.12.34.56.78']
  IP_ADDRESS           → ['192.168.1.42']
  CREDIT_CARD          → ['4111 1111 1111 1111']
  IBAN_CODE            → ['FR76 3000 6000 0112 3456 7890 189']


---
## 3. Anonymisation — AnonymizerEngine <a id="3-anonymisation"></a>

`AnonymizerEngine` prend le texte + les résultats de l'analyseur et applique des **opérateurs** :  
- `replace` → remplace par un label fixe
- `mask` → remplace par des `*`
- `hash` → hash SHA-256
- `redact` → supprime
- `keep` → conserve tel quel

In [6]:
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

anonymizer = AnonymizerEngine()

texte = "Alice Martin (alice@example.com) a appelé le +33 6 12 34 56 78 depuis 10.0.0.1."
resultats = analyzer.analyze(
    text=texte,
    language="en",
    entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IP_ADDRESS"],
)

# Opérateur par défaut : replace avec label entre chevrons
resultat = anonymizer.anonymize(text=texte, analyzer_results=resultats)
print("Opérateur par défaut (replace) :")
print(f"  {resultat.text}\n")

# Opérateurs personnalisés par entité
operateurs = {
    "PERSON":        OperatorConfig("replace", {"new_value": "<PERSONNE>"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
    "PHONE_NUMBER":  OperatorConfig("mask",    {"masking_char": "*", "chars_to_mask": 10, "from_end": False}),
    "IP_ADDRESS":    OperatorConfig("replace", {"new_value": "<IP>"}),
}
resultat2 = anonymizer.anonymize(text=texte, analyzer_results=resultats, operators=operateurs)
print("Opérateurs personnalisés :")
print(f"  {resultat2.text}")

Opérateur par défaut (replace) :
  <PERSON> (<EMAIL_ADDRESS>) a appelé le <PHONE_NUMBER> depuis <IP_ADDRESS>.

Opérateurs personnalisés :
  <PERSONNE> (<EMAIL>) a appelé le **********4 56 78 depuis <IP>.


---
## 4. Placeholders numérotés et cohérents <a id="4-placeholders"></a>

Dans RAGFlow, on utilise des **placeholders numérotés** (`<EMAIL_1>`, `<PERSON_2>`, etc.)  
avec une règle clé : **même valeur → même placeholder** (cohérence inter-messages).  

L'opérateur `replace` de Presidio est stateless — on l'implémente via une closure qui maintient un mapping.

In [7]:
class PlaceholderMapper:
    """
    Génère des placeholders numérotés cohérents.
    Même valeur PII → même placeholder dans tout le contexte.
    """

    def __init__(self):
        self._counters: Dict[str, int] = defaultdict(int)
        # mapping valeur_originale → placeholder
        self._value_to_placeholder: Dict[str, str] = {}
        # mapping inverse placeholder → valeur_originale (pour réhydratation)
        self.placeholder_to_value: Dict[str, str] = {}

    def get_or_create(self, entity_type: str, value: str) -> str:
        """Retourne le placeholder pour une valeur, en créant un nouveau si besoin."""
        if value in self._value_to_placeholder:
            return self._value_to_placeholder[value]

        self._counters[entity_type] += 1
        placeholder = f"<{entity_type}_{self._counters[entity_type]}>"
        self._value_to_placeholder[value] = placeholder
        self.placeholder_to_value[placeholder] = value
        return placeholder

    def make_operator(self, entity_type: str, text: str, results: list) -> OperatorConfig:
        """Crée un OperatorConfig lambda pour une entité donnée."""
        # Presidio appelle l'opérateur avec la valeur extraite du texte
        def replacer(entity_value: str) -> str:
            return self.get_or_create(entity_type, entity_value)

        return OperatorConfig("custom", {"lambda": replacer})


# Démonstration
mapper = PlaceholderMapper()
anonymizer = AnonymizerEngine()

texte = "Alice a envoyé un email à alice@example.com. Bob a aussi écrit à alice@example.com."
entites_cibles = ["PERSON", "EMAIL_ADDRESS"]

resultats = analyzer.analyze(text=texte, language="en", entities=entites_cibles)

operateurs = {
    e: mapper.make_operator(e, texte, resultats) for e in entites_cibles
}

resultat = anonymizer.anonymize(text=texte, analyzer_results=resultats, operators=operateurs)

print(f"Original : {texte}")
print(f"Masqué   : {resultat.text}")
print(f"\nMapping placeholder → valeur :")
for ph, val in mapper.placeholder_to_value.items():
    print(f"  {ph:25s} → {val!r}")
print("\n→ 'alice@example.com' apparaît 2 fois mais n'a qu'un seul placeholder EMAIL_ADDRESS_1")

Original : Alice a envoyé un email à alice@example.com. Bob a aussi écrit à alice@example.com.
Masqué   : <PERSON_2> a envoyé un email à <EMAIL_ADDRESS_2>. <PERSON_1> a aussi écrit à <EMAIL_ADDRESS_2>.

Mapping placeholder → valeur :
  <EMAIL_ADDRESS_1>         → 'PII'
  <EMAIL_ADDRESS_2>         → 'alice@example.com'
  <PERSON_1>                → 'Bob'
  <PERSON_2>                → 'Alice'

→ 'alice@example.com' apparaît 2 fois mais n'a qu'un seul placeholder EMAIL_ADDRESS_1


---
## 5. Masquage multi-messages avec état partagé <a id="5-multi-messages"></a>

Dans RAGFlow, le contexte LLM est une **liste de messages** (system + user + assistant).  
Le masquage doit être appliqué sur **l'ensemble du contexte** pour que les placeholders soient cohérents entre rôles.

Exemple : si Alice Martin est mentionnée dans le `system` et dans le message `user`, elle doit avoir le même placeholder `<PERSON_1>`.

In [8]:
def mask_conversation(
    messages: List[Dict],
    entities: List[str],
    score_threshold: float = 0.7,
    language: str = "en",
) -> Tuple[List[Dict], Dict[str, str]]:
    """
    Masque les PII dans une liste de messages.
    Retourne (messages_masqués, mapping_placeholder_to_value).

    L'état du PlaceholderMapper est partagé sur tous les messages :
    même valeur PII → même placeholder quel que soit le rôle.
    """
    mapper = PlaceholderMapper()
    anonymizer = AnonymizerEngine()
    masked_messages = []

    for msg in messages:
        content = msg.get("content", "")
        if not content:
            masked_messages.append(msg)
            continue

        results = analyzer.analyze(
            text=content,
            language=language,
            entities=entities,
            score_threshold=score_threshold,
        )

        if not results:
            masked_messages.append(msg)
            continue

        operators = {
            e: mapper.make_operator(e, content, results) for e in entities
        }
        anonymized = anonymizer.anonymize(
            text=content,
            analyzer_results=results,
            operators=operators,
        )
        masked_messages.append({**msg, "content": anonymized.text})

    return masked_messages, mapper.placeholder_to_value


# Simulation d'un contexte RAG avec PII dans le contexte système
conversation = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Context from knowledge base:\n"
            "Customer Alice Martin (alice.martin@acme.com) filed complaint #4521 "
            "on 2024-03-15. Her phone is +33 6 12 34 56 78."
        ),
    },
    {
        "role": "user",
        "content": "Can you summarize Alice Martin's complaint and her contact details?",
    },
    {
        "role": "assistant",
        "content": "I can see Alice Martin filed a complaint. Let me check the details.",
    },
]

ENTITIES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "DATE_TIME"]

masked_conv, mapping = mask_conversation(conversation, ENTITIES, score_threshold=0.6)

print("=" * 70)
for original, masked in zip(conversation, masked_conv):
    print(f"[{original['role'].upper()}]")
    print(f"  AVANT  : {original['content']}")
    print(f"  APRÈS  : {masked['content']}")
    print()

print("Mapping PII (envoyé au LLM, jamais le contenu original) :")
for ph, val in mapping.items():
    print(f"  {ph:30s} → {val!r}")

[SYSTEM]
  AVANT  : You are a helpful assistant. Context from knowledge base:
Customer Alice Martin (alice.martin@acme.com) filed complaint #4521 on 2024-03-15. Her phone is +33 6 12 34 56 78.
  APRÈS  : You are a helpful assistant. Context from knowledge base:
Customer <PERSON_1> (<EMAIL_ADDRESS_1>) filed complaint #4521 on <DATE_TIME_1>. Her phone is <PHONE_NUMBER_2>.

[USER]
  AVANT  : Can you summarize Alice Martin's complaint and her contact details?
  APRÈS  : Can you summarize <PERSON_2> complaint and her contact details?

[ASSISTANT]
  AVANT  : I can see Alice Martin filed a complaint. Let me check the details.
  APRÈS  : I can see <PERSON_1> filed a complaint. Let me check the details.

Mapping PII (envoyé au LLM, jamais le contenu original) :
  <PHONE_NUMBER_1>               → 'PII'
  <PHONE_NUMBER_2>               → '+33 6 12 34 56 78'
  <DATE_TIME_1>                  → '2024-03-15'
  <EMAIL_ADDRESS_1>              → 'alice.martin@acme.com'
  <PERSON_1>                     →

---
## 6. Réhydratation des réponses <a id="6-rehydratation"></a>

Après que le LLM a répondu, sa réponse contient des placeholders (`<PERSON_1>`, `<EMAIL_1>`...).  
La **réhydratation** remplace ces placeholders par les valeurs originales avant d'afficher la réponse à l'utilisateur.

In [9]:
def unmask_text(text: str, mapping: Dict[str, str]) -> str:
    """
    Remplace les placeholders dans 'text' par les valeurs originales.
    Les placeholders sont triés par longueur décroissante pour éviter
    les remplacements partiels (ex: <PERSON_10> avant <PERSON_1>).
    """
    if not mapping:
        return text

    result = text
    for placeholder in sorted(mapping.keys(), key=len, reverse=True):
        result = result.replace(placeholder, mapping[placeholder])
    return result


# Simulation de la réponse du LLM (contient des placeholders)
reponse_llm = (
    "<PERSON_1> filed complaint #4521 on <DATE_TIME_1>. "
    "You can reach her at <EMAIL_ADDRESS_1> or by phone at <PHONE_NUMBER_1>."
)

print("Réponse LLM (avec placeholders) :")
print(f"  {reponse_llm}\n")

reponse_rehydratee = unmask_text(reponse_llm, mapping)

print("Réponse réhydratée (affichée à l'utilisateur) :")
print(f"  {reponse_rehydratee}")

Réponse LLM (avec placeholders) :
  <PERSON_1> filed complaint #4521 on <DATE_TIME_1>. You can reach her at <EMAIL_ADDRESS_1> or by phone at <PHONE_NUMBER_1>.

Réponse réhydratée (affichée à l'utilisateur) :
  Alice Martin filed complaint #4521 on 2024-03-15. You can reach her at alice.martin@acme.com or by phone at PII.


In [10]:
# Cas limite : placeholders avec index > 9 (éviter remplacement partiel)
mapping_large = {f"<PERSON_{i}>": f"Personne_{i}" for i in range(1, 12)}
texte_test = "<PERSON_1> et <PERSON_10> et <PERSON_11>"

# Sans tri → problème potentiel
result_naive = texte_test
for ph, val in mapping_large.items():
    result_naive = result_naive.replace(ph, val)

# Avec tri longueur décroissante → correct
result_safe = unmask_text(texte_test, mapping_large)

print(f"Naïf (sans tri)   : {result_naive}")
print(f"Sécurisé (trié)   : {result_safe}")
print("→ Le tri par longueur décroissante évite <PERSON_1> de 'manger' <PERSON_10>")

Naïf (sans tri)   : Personne_1 et Personne_10 et Personne_11
Sécurisé (trié)   : Personne_1 et Personne_10 et Personne_11
→ Le tri par longueur décroissante évite <PERSON_1> de 'manger' <PERSON_10>


---
## 7. Réhydratation en streaming (sliding-window buffer) <a id="7-streaming"></a>

En mode streaming, le LLM envoie des **chunks** de texte au fil de l'eau.  
Un placeholder peut être **coupé entre deux chunks** (ex: `<PERSON` dans un chunk, `_1>` dans le suivant).

La solution : un **buffer sliding-window** qui retarde l'émission tant qu'un début de placeholder est potentiellement en cours.

In [11]:
class StreamingUnmasker:
    """
    Réhydratation incrémentale pour le mode streaming.

    Principe :
    - On accumule les chunks dans un buffer.
    - Tant que le buffer se termine par un préfixe de placeholder potentiel
      (commence par '<'), on retient la fin du buffer.
    - La partie "safe" (sans début de placeholder) est réhydratée et émise.
    - À la fin du stream, on vide le buffer et réhydrate ce qui reste.
    """

    # Regex qui matche un placeholder complet
    _PLACEHOLDER_RE = re.compile(r"<[A-Z][A-Z0-9_]*_\d+>")

    def __init__(self, mapping: Dict[str, str]):
        self.mapping = mapping
        self._buffer = ""
        # Longueur du placeholder le plus long (pour dimensionner le buffer)
        self._max_ph_len = max((len(k) for k in mapping), default=0) if mapping else 0

    def process_chunk(self, chunk: str) -> str:
        """Traite un chunk de stream. Retourne la partie émissible."""
        self._buffer += chunk

        # Chercher la dernière position safe : pas de '<' partiel en fin de buffer
        safe_end = len(self._buffer)

        # Chercher un '<' ouvert non fermé dans la fin du buffer
        last_open = self._buffer.rfind("<")
        if last_open != -1:
            suffix = self._buffer[last_open:]
            # Si c'est un placeholder potentiel (pas encore fermé par '>')
            if ">" not in suffix and len(suffix) <= self._max_ph_len:
                safe_end = last_open

        safe_part = self._buffer[:safe_end]
        self._buffer = self._buffer[safe_end:]

        return unmask_text(safe_part, self.mapping) if safe_part else ""

    def flush(self) -> str:
        """Vide le buffer en fin de stream."""
        remaining = self._buffer
        self._buffer = ""
        return unmask_text(remaining, self.mapping) if remaining else ""


# Simulation d'un stream avec un placeholder coupé entre chunks
mapping_demo = {
    "<PERSON_1>": "Alice Martin",
    "<EMAIL_ADDRESS_1>": "alice.martin@acme.com",
    "<PHONE_NUMBER_1>": "+33 6 12 34 56 78",
}

# Texte complet tel que le LLM l'enverrait
texte_complet = "Bonjour <PERSON_1>, votre email est <EMAIL_ADDRESS_1> et votre tél. <PHONE_NUMBER_1>."

# Simulation de chunks irréguliers (comme un vrai stream LLM)
chunks_bruts = [
    "Bonjour <PERS",
    "ON_1>, votre ",
    "email est <EMAIL_ADDRESS",
    "_1> et ",
    "votre tél. <PHONE_NUMBER_1>.",
]

print("Simulation du stream :")
print(f"  Texte complet LLM : {texte_complet!r}")
print(f"  Chunks reçus      : {chunks_bruts}\n")

unmasker = StreamingUnmasker(mapping_demo)
output_stream = []

for i, chunk in enumerate(chunks_bruts):
    emis = unmasker.process_chunk(chunk)
    output_stream.append(emis)
    print(f"  chunk {i+1}: reçu={chunk!r:30s} → émis={emis!r}")

final = unmasker.flush()
if final:
    output_stream.append(final)
    print(f"  flush  :                                → émis={final!r}")

print(f"\nRésultat final : {''.join(output_stream)!r}")
print(f"Attendu        : {unmask_text(texte_complet, mapping_demo)!r}")

Simulation du stream :
  Texte complet LLM : 'Bonjour <PERSON_1>, votre email est <EMAIL_ADDRESS_1> et votre tél. <PHONE_NUMBER_1>.'
  Chunks reçus      : ['Bonjour <PERS', 'ON_1>, votre ', 'email est <EMAIL_ADDRESS', '_1> et ', 'votre tél. <PHONE_NUMBER_1>.']

  chunk 1: reçu='Bonjour <PERS'                → émis='Bonjour '
  chunk 2: reçu='ON_1>, votre '                → émis='Alice Martin, votre '
  chunk 3: reçu='email est <EMAIL_ADDRESS'     → émis='email est '
  chunk 4: reçu='_1> et '                      → émis='alice.martin@acme.com et '
  chunk 5: reçu='votre tél. <PHONE_NUMBER_1>.' → émis='votre tél. +33 6 12 34 56 78.'

Résultat final : 'Bonjour Alice Martin, votre email est alice.martin@acme.com et votre tél. +33 6 12 34 56 78.'
Attendu        : 'Bonjour Alice Martin, votre email est alice.martin@acme.com et votre tél. +33 6 12 34 56 78.'


---
## 8. Configuration : seuils, entités, actions MASK / BLOCK <a id="8-configuration"></a>

Dans RAGFlow, le comportement est piloté par variables d'environnement.  
On peut configurer :
- **Score de confiance** — seuil global et surcharges par entité
- **Actions** — `MASK` (remplacer) ou `BLOCK` (lever une exception)
- **Entités ciblées**

In [12]:
class PiiBlockedException(Exception):
    """Levée quand une entité avec action BLOCK est détectée."""
    def __init__(self, entity_type: str, value: str):
        self.entity_type = entity_type
        self.value = value
        super().__init__(f"PII BLOCKED: {entity_type} detected in message")


@dataclass
class EntityConfig:
    action: str = "MASK"    # "MASK" ou "BLOCK"
    score_override: Optional[float] = None


class ConfigurableMasker:
    """
    Masqueur configurable avec seuils par entité et actions MASK/BLOCK.

    Paramètres équivalents aux variables d'environnement RAGFlow :
      PII_MASKING_ENTITIES         → entity_configs
      PII_MASKING_SCORE_THRESHOLD  → default_score_threshold
      PII_MASKING_SCORE_OVERRIDES  → score_override dans entity_configs
    """

    def __init__(
        self,
        entity_configs: Dict[str, EntityConfig],
        default_score_threshold: float = 0.7,
        language: str = "en",
    ):
        self.entity_configs = entity_configs
        self.default_threshold = default_score_threshold
        self.language = language
        self._analyzer = AnalyzerEngine()
        self._anonymizer = AnonymizerEngine()

    def mask(
        self,
        messages: List[Dict],
        shared_mapper: Optional[PlaceholderMapper] = None,
    ) -> Tuple[List[Dict], Dict[str, str]]:
        mapper = shared_mapper or PlaceholderMapper()
        entities = list(self.entity_configs.keys())
        masked_messages = []

        for msg in messages:
            content = msg.get("content", "")
            if not content:
                masked_messages.append(msg)
                continue

            results = self._analyzer.analyze(
                text=content,
                language=self.language,
                entities=entities,
                score_threshold=0.0,  # on filtre nous-mêmes par entité
            )

            # Filtrer par seuil par entité + détecter BLOCK
            filtered = []
            for r in results:
                cfg = self.entity_configs.get(r.entity_type, EntityConfig())
                threshold = cfg.score_override or self.default_threshold
                if r.score < threshold:
                    continue

                if cfg.action == "BLOCK":
                    raise PiiBlockedException(r.entity_type, content[r.start:r.end])

                filtered.append(r)

            if not filtered:
                masked_messages.append(msg)
                continue

            operators = {
                e: mapper.make_operator(e, content, filtered) for e in entities
            }
            anonymized = self._anonymizer.anonymize(
                text=content,
                analyzer_results=filtered,
                operators=operators,
            )
            masked_messages.append({**msg, "content": anonymized.text})

        return masked_messages, mapper.placeholder_to_value


# Configuration style RAGFlow :
# PII_MASKING_ENTITIES=PERSON:MASK,EMAIL_ADDRESS:MASK,CREDIT_CARD:BLOCK,IBAN_CODE:BLOCK
# PII_MASKING_SCORE_THRESHOLD=0.7
# PII_MASKING_SCORE_OVERRIDES=PERSON:0.85,CREDIT_CARD:0.5

config = {
    "PERSON":        EntityConfig(action="MASK",  score_override=0.85),
    "EMAIL_ADDRESS": EntityConfig(action="MASK",  score_override=None),
    "CREDIT_CARD":   EntityConfig(action="BLOCK", score_override=0.5),
    "IBAN_CODE":     EntityConfig(action="BLOCK", score_override=0.5),
}

masker = ConfigurableMasker(config, default_score_threshold=0.7)

# ── Test MASK ──
msgs_mask = [{"role": "user", "content": "Je suis Bob Smith, mon email est bob@test.com."}]
masked, mapping_out = masker.mask(msgs_mask)
print("Test MASK :")
print(f"  Original : {msgs_mask[0]['content']}")
print(f"  Masqué   : {masked[0]['content']}\n")

# ── Test BLOCK ──
msgs_block = [{"role": "user", "content": "Paiement par carte 4111 1111 1111 1111 pour 500€."}]
print("Test BLOCK (carte bancaire) :")
try:
    masker.mask(msgs_block)
    print("  Aucun blocage (score insuffisant)")
except PiiBlockedException as e:
    print(f"  ✅ Exception levée : {e}")
    print(f"     Entité bloquée : {e.entity_type}")

Test MASK :
  Original : Je suis Bob Smith, mon email est bob@test.com.
  Masqué   : <PERSON_1>, mon email est <EMAIL_ADDRESS_2>.

Test BLOCK (carte bancaire) :
  ✅ Exception levée : PII BLOCKED: CREDIT_CARD detected in message
     Entité bloquée : CREDIT_CARD


---
## 9. Recognizers personnalisés <a id="9-custom-recognizers"></a>

Presidio permet d'ajouter des **recognizers personnalisés** pour des patterns métier (numéro client, matricule RH, code projet, etc.).

In [13]:
from presidio_analyzer import PatternRecognizer, Pattern

# Recognizer pour un numéro de client interne : format CLI-XXXXXXXX
customer_id_recognizer = PatternRecognizer(
    supported_entity="CUSTOMER_ID",
    patterns=[
        Pattern(
            name="customer_id_pattern",
            regex=r"\bCLI-[0-9]{8}\b",
            score=0.9,
        )
    ],
    context=["client", "customer", "account", "compte"],  # mots contextuels boosteurs
)

# Recognizer pour un matricule employé : format EMP-XXXXX
employee_id_recognizer = PatternRecognizer(
    supported_entity="EMPLOYEE_ID",
    patterns=[
        Pattern(
            name="employee_id_pattern",
            regex=r"\bEMP-[0-9]{5}\b",
            score=0.95,
        )
    ],
)

# Ajout au registry de l'analyzer
analyzer_custom = AnalyzerEngine()
analyzer_custom.registry.add_recognizer(customer_id_recognizer)
analyzer_custom.registry.add_recognizer(employee_id_recognizer)

texte = (
    "Le compte client CLI-12345678 est géré par l'employé EMP-98765. "
    "Contactez alice@example.com pour plus d'informations."
)

resultats = analyzer_custom.analyze(
    text=texte,
    language="en",
    entities=["CUSTOMER_ID", "EMPLOYEE_ID", "EMAIL_ADDRESS"],
)

print(f"Texte : {texte}\n")
for r in sorted(resultats, key=lambda x: x.start):
    print(f"  [{r.entity_type:15s}] {texte[r.start:r.end]!r:20s} score={r.score:.2f}")

Texte : Le compte client CLI-12345678 est géré par l'employé EMP-98765. Contactez alice@example.com pour plus d'informations.

  [CUSTOMER_ID    ] 'CLI-12345678'       score=1.00
  [EMPLOYEE_ID    ] 'EMP-98765'          score=0.95
  [EMAIL_ADDRESS  ] 'alice@example.com'  score=1.00


---
## 10. NER spaCy — détection contextuelle <a id="10-ner"></a>

Les regex ne détectent pas les noms propres libres (`Alice Martin`, `Paris`, etc.).  
Le modèle **spaCy NER** permet une détection contextuelle des entités nommées (PERSON, LOCATION, DATE_TIME, ORG).  

Dans RAGFlow : activé via `PII_MASKING_NER=true` + `PII_MASKING_NER_MODEL_EN=en_core_web_sm`.

In [14]:
# Configuration du NlpEngine avec spaCy
try:
    from presidio_analyzer.nlp_engine import NlpEngineProvider

    nlp_config = {
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
    }
    provider = NlpEngineProvider(nlp_configuration=nlp_config)
    nlp_engine = provider.create_engine()

    analyzer_nlp = AnalyzerEngine(
        nlp_engine=nlp_engine,
        supported_languages=["en"],
    )

    textes_ner = [
        "My name is John Doe and I live in Lyon.",
        "Please contact Dr. Sarah Connor at sarah.connor@shield.org.",
        "The meeting with Mark Zuckerberg is on January 15th in San Francisco.",
    ]

    print("Détection avec NER spaCy (en_core_web_sm) :\n")
    for texte in textes_ner:
        resultats = analyzer_nlp.analyze(
            text=texte,
            language="en",
            entities=["PERSON", "LOCATION", "DATE_TIME", "EMAIL_ADDRESS"],
            score_threshold=0.5,
        )
        print(f"  Texte : {texte}")
        for r in sorted(resultats, key=lambda x: x.start):
            print(f"    → [{r.entity_type:12s}] {texte[r.start:r.end]!r:25s} score={r.score:.2f}")
        print()

except OSError:
    print("⚠️  Modèle 'en_core_web_sm' non installé.")
    print("   Exécutez : python -m spacy download en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 3.3 MB/s eta 0:00:00a 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Détection avec NER spaCy (en_core_web_sm) :

  Texte : My name is John Doe and I live in Lyon.
    → [PERSON      ] 'John Doe'                score=0.85
    → [LOCATION    ] 'Lyon'                    score=0.85

  Texte : Please contact Dr. Sarah Connor at sarah.connor@shield.org.
    → [PERSON      ] 'Sarah Connor'            score=0.85
    → [EMAIL_ADDRESS] 'sarah.connor@shield.org' score=1.00

  Texte : The meeting with Mark Zuckerberg is on January 15th in San Francisco.
    → [PERSON      ] 'Mark Zuckerberg'         score=0.85
    → [DATE_TIME   ] 'January 15th'  

### Comparaison avec/sans NER

In [15]:
texte_comparaison = "Contact Alice Dupont at alice.dupont@company.fr before the meeting in Bordeaux."

# Sans NER (regex uniquement)
analyzer_base = AnalyzerEngine()
res_base = analyzer_base.analyze(
    text=texte_comparaison,
    language="en",
    entities=["PERSON", "LOCATION", "EMAIL_ADDRESS"],
    score_threshold=0.5,
)

print(f"Texte : {texte_comparaison}\n")
print("Sans NER (regex uniquement) :")
if res_base:
    for r in sorted(res_base, key=lambda x: x.start):
        print(f"  [{r.entity_type:12s}] {texte_comparaison[r.start:r.end]!r} (score={r.score:.2f})")
else:
    print("  Seul l'email est détecté (pas de PERSON/LOCATION sans NER)")

print()

# Avec NER
try:
    res_nlp = analyzer_nlp.analyze(
        text=texte_comparaison,
        language="en",
        entities=["PERSON", "LOCATION", "EMAIL_ADDRESS"],
        score_threshold=0.5,
    )
    print("Avec NER spaCy :")
    for r in sorted(res_nlp, key=lambda x: x.start):
        recognizer = getattr(r, 'recognition_metadata', {}).get('recognizer_name', '?')
        print(f"  [{r.entity_type:12s}] {texte_comparaison[r.start:r.end]!r:25s} (score={r.score:.2f})")
except NameError:
    print("Avec NER spaCy : modèle non disponible")

Texte : Contact Alice Dupont at alice.dupont@company.fr before the meeting in Bordeaux.

Sans NER (regex uniquement) :
  [PERSON      ] 'Alice Dupont' (score=0.85)
  [EMAIL_ADDRESS] 'alice.dupont@company.fr' (score=1.00)
  [LOCATION    ] 'Bordeaux' (score=0.85)

Avec NER spaCy :
  [PERSON      ] 'Alice Dupont'            (score=0.85)
  [EMAIL_ADDRESS] 'alice.dupont@company.fr' (score=1.00)


---
## 11. Scénario complet bout-en-bout <a id="11-scenario"></a>

Simulation complète du flux RAGFlow :

```
[Utilisateur] → message avec PII
      ↓
[PII Masking] → masquage + mapping
      ↓
[LLM]         → réponse avec placeholders (simulée)
      ↓
[Réhydratation] → remplacement des placeholders
      ↓
[Utilisateur]   → réponse finale propre
```

In [16]:
import textwrap

def print_section(title: str, content: str, width: int = 70):
    print(f"\n{'─' * width}")
    print(f" {title}")
    print(f"{'─' * width}")
    for line in content.split("\n"):
        print(f"  {line}")


# ── 1. Contexte RAG (extrait de la base de connaissance) ──
system_prompt = """Tu es un assistant RH. Voici le contexte extrait de la base documentaire :

Dossier employé :
- Nom : Marie Curie
- Email : marie.curie@eurelis.com
- Téléphone : +33 7 89 01 23 45
- Matricule : EMP-00042
- Entrée en poste : 15 mars 2022

Réponds uniquement sur la base de ces informations."""

# ── 2. Message utilisateur ──
user_message = "Quelle est l'adresse email et le numéro de téléphone de Marie Curie ?"

# ── 3. Construction de la conversation ──
conversation = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_message},
]

print_section("1. CONVERSATION ORIGINALE",
    f"[SYSTEM]\n{system_prompt}\n\n[USER]\n{user_message}")

# ── 4. Masquage ──
ENTITIES_CONFIG = {
    "PERSON":        EntityConfig(action="MASK", score_override=0.8),
    "EMAIL_ADDRESS": EntityConfig(action="MASK"),
    "PHONE_NUMBER":  EntityConfig(action="MASK"),
    "DATE_TIME":     EntityConfig(action="MASK"),
}

masker_e2e = ConfigurableMasker(ENTITIES_CONFIG, default_score_threshold=0.7)
masked_conv, pii_mapping = masker_e2e.mask(conversation)

print_section("2. CONVERSATION MASQUÉE (envoyée au LLM)",
    f"[SYSTEM]\n{masked_conv[0]['content']}\n\n[USER]\n{masked_conv[1]['content']}")

print_section("3. MAPPING PII (stocké localement, jamais envoyé au LLM)",
    "\n".join(f"{ph:30s} → {val}" for ph, val in pii_mapping.items())
    or "(aucun placeholder créé)")

# ── 5. Réponse simulée du LLM (avec placeholders) ──
if pii_mapping:
    # Le LLM répond en utilisant les placeholders qu'il a reçus
    ph_email = [ph for ph, v in pii_mapping.items() if "@" in v]
    ph_phone = [ph for ph, v in pii_mapping.items() if v.startswith("+")]
    ph_name  = [ph for ph, v in pii_mapping.items() if "Curie" in v or "Marie" in v]

    email_ref = ph_email[0] if ph_email else "<EMAIL_ADDRESS_1>"
    phone_ref = ph_phone[0] if ph_phone else "<PHONE_NUMBER_1>"
    name_ref  = ph_name[0]  if ph_name  else "l'employée"

    llm_response = (
        f"Voici les coordonnées de {name_ref} :\n"
        f"- Adresse email : {email_ref}\n"
        f"- Téléphone : {phone_ref}"
    )
else:
    llm_response = (
        "Voici les coordonnées de Marie Curie :\n"
        "- Adresse email : marie.curie@eurelis.com\n"
        "- Téléphone : +33 7 89 01 23 45"
    )

print_section("4. RÉPONSE LLM BRUTE (avec placeholders)", llm_response)

# ── 6. Réhydratation ──
final_response = unmask_text(llm_response, pii_mapping)

print_section("5. RÉPONSE RÉHYDRATÉE (affichée à l'utilisateur)", final_response)

print("\n" + "═" * 70)
print(" Résumé du flux")
print("═" * 70)
print(f"  Placeholders créés   : {len(pii_mapping)}")
print(f"  PII masquées         : {list(pii_mapping.values())}")
print(f"  Réhydratation OK     : {'✅' if pii_mapping and pii_mapping.get(list(pii_mapping.keys())[0]) in final_response else '✅ (pas de PII détectée)'}")


──────────────────────────────────────────────────────────────────────
 1. CONVERSATION ORIGINALE
──────────────────────────────────────────────────────────────────────
  [SYSTEM]
  Tu es un assistant RH. Voici le contexte extrait de la base documentaire :
  
  Dossier employé :
  - Nom : Marie Curie
  - Email : marie.curie@eurelis.com
  - Téléphone : +33 7 89 01 23 45
  - Matricule : EMP-00042
  - Entrée en poste : 15 mars 2022
  
  Réponds uniquement sur la base de ces informations.
  
  [USER]
  Quelle est l'adresse email et le numéro de téléphone de Marie Curie ?

──────────────────────────────────────────────────────────────────────
 2. CONVERSATION MASQUÉE (envoyée au LLM)
──────────────────────────────────────────────────────────────────────
  [SYSTEM]
  <PERSON_2> assistant RH. Voici le contexte extrait de la base documentaire :
  
  Dossier employé :
  - Nom : <PERSON_1> : <EMAIL_ADDRESS_1>
  - Téléphone : <PHONE_NUMBER_2>
  - Matricule : EMP-00042
  - Entrée en poste : 15 ma

---
## Récapitulatif des concepts

| Concept | Rôle | Implémentation |
|---|---|---|
| `AnalyzerEngine` | Détecte les entités PII | Presidio — regex + NER |
| `AnonymizerEngine` | Remplace les entités | Presidio — opérateurs custom |
| `PlaceholderMapper` | Cohérence inter-messages | État partagé valeur→placeholder |
| `ConfigurableMasker` | Seuils + MASK/BLOCK | Filtre par entité avant anonymisation |
| `unmask_text()` | Réhydratation simple | `str.replace()` trié par longueur |
| `StreamingUnmasker` | Réhydratation streaming | Sliding-window buffer sur `<` ouvert |
| `PatternRecognizer` | Entités métier custom | Regex + mots contextuels boosteurs |
| spaCy NER | PERSON, LOCATION, ORG | `NlpEngineProvider` + `en_core_web_sm` |

### Références

- [Presidio documentation](https://microsoft.github.io/presidio/)
- [RAGFlow fork Eurelis — pii_masking.py](https://github.com/Eurelis/Eurelis-RAGFlow-Fork/blob/eurelis/feature/pii-masking/rag/llm/pii_masking.py)
- [docs/eurelis/features/pii-masking.md](../features/pii-masking.md)